# 向量库知识智能体 (DB Agent RAG)

---

本教程演示如何将 `files` 目录下的 txt 文件按空行分段，写入 Chroma 内存向量库，并通过 ReAct Agent 检索回答。

### 数据准备提示
1. 请在当前目录下创建一个 `files` 文件夹。
2. 在其中放入 `.txt` 文件。
3. **注意**：若是 Excel 数据，需额外写脚本将列表头转为以下格式（空行分割），问答效果更准：

```text
问题：<问题内容>
答案：<答案内容>

（此处为空行）
```

## 1. 导入依赖与加载环境

确保目录下存在 `.env` 文件，包含 `API_KEY` 和 `BASE_URL`。

In [37]:
import os
import re
from pathlib import Path
from typing import Iterable

from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma
import chromadb
from chromadb.config import Settings
from langchain.agents import create_agent
from langchain.tools import tool

# 加载模型配置
_ = load_dotenv()

In [39]:
from importlib.metadata import version
print("dashscope version: ", version("dashscope"))
print("langchain version: ", version("langchain"))
print("langgraph version: ", version("langgraph"))
print("langchain_openai version: ", version("langchain_openai"))
print("langchain_chroma version: ", version("langchain_chroma"))
print("chromadb version: ", version("chromadb"))

dashscope version:  1.24.6
langchain version:  1.2.10
langgraph version:  1.0.9
langchain_openai version:  1.1.11
langchain_chroma version:  1.1.0
chromadb version:  1.5.5


## 2. 配置大模型与客户端

In [3]:
# 配置大模型
llm = ChatOpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
    model="qwen3-coder-plus",
    temperature=0,
)

# 创建 OpenAI 客户端 (用于 Embeddings)
client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
)

## 3. 定义 DashScope Embeddings

自定义 Embeddings 类以适配 LangChain 接口。

In [4]:
class TextEmbeddings(Embeddings):
    """DashScope 兼容的 Embeddings 封装。"""

    def __init__(self, model: str = "text-embedding-v4", dimensions: int = 1024):
        self.model = model
        self.dimensions = dimensions

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        vectors: list[list[float]] = []
        for i in range(0, len(texts), 10):
            chunk = texts[i : i + 10]
            response = client.embeddings.create(
                model=self.model,
                input=chunk,
                dimensions=self.dimensions,
            )
            vectors.extend([item.embedding for item in response.data])
        return vectors

    def embed_query(self, text: str) -> list[float]:
        response = client.embeddings.create(
            model=self.model,
            input=[text],
            dimensions=self.dimensions,
        )
        return response.data[0].embedding

## 4. 文档加载逻辑

读取目录下的 txt 文件，并使用**空行**进行切分。

In [5]:
def load_txt_documents(data_dir: Path) -> list[Document]:
    """读取目录下的 txt 文件并按空行分割为 Document。"""

    def split_on_blank(text: str) -> Iterable[str]:
        for block in re.split(r"\n\s*\n", text):
            cleaned = block.strip()
            if cleaned:
                yield cleaned

    documents: list[Document] = []
    # 检查目录是否存在
    if not data_dir.exists():
        print(f"Warning: 目录 {data_dir} 不存在")
        return []

    for path in sorted(data_dir.glob("*.txt")):
        content = path.read_text(encoding="utf-8")
        for idx, part in enumerate(split_on_blank(content)):
            documents.append(
                Document(
                    page_content=part,
                    metadata={"source": path.name, "chunk_id": idx},
                )
            )
    if not documents:
        print(f"目录 {data_dir} 下未找到 txt 文档")
    return documents

## 5. 构建向量数据库

连接到 Chroma 服务并写入数据。

In [6]:
def build_vector_store(data_dir: Path | None = None) -> Chroma:
    """读取 txt 文件并构建内存向量库。"""
    # 指向当前目录下的 files
    target_dir = data_dir or (Path.cwd() / "files")
    documents = load_txt_documents(target_dir)

    print(f"成功加载 {len(documents)} 个文档到向量库")

    # 文档的向量化
    embeddings = TextEmbeddings()
    
    # 配置 Chroma 设置
    chroma_settings = Settings(
        persist_directory="./db/chroma_data",  # 指定持久化目录
        is_persistent=True  # 1.0+ 版本默认即为 True
    )
    
    vector_store = Chroma(
        collection_name="test_collection",
        embedding_function=embeddings,
        client_settings=chroma_settings

    )
    
    # 清空集合（可选，防止重复数据堆积）
    try:
        existing_ids = vector_store.get()["ids"]
        if existing_ids:
            vector_store.delete(ids=existing_ids)
    except Exception as e:
        print(f"连接或清理数据库时提示: {e}")
    
    # 添加文档
    if documents:
        # 向量化
        _ = vector_store.add_documents(documents)
    
    return vector_store

## 6. 创建 ReAct Agent

定义检索工具并初始化 Agent。

In [26]:
# 备注：新版代码对reranker的实现做了调整，用TextReRank替换 DashScopeRerank
# 解决 DashScopeRerank 因为鉴权返回为None的问题

import dashscope
from dashscope import TextReRank

dashscope.api_key = os.environ["API_KEY"]

In [35]:
def create_react_agent_wrapper(vector_store: Chroma):
    """基于给定向量库创建带检索工具的 ReAct Agent。"""

    @tool(response_format="content_and_artifact")
    def retrieve_context(query: str):
        """基于向量库检索与问题最相关的文本片段并排序。"""
        
        # 检索
        retrieved = vector_store.similarity_search(query, k=5)
        print("【DEBUG】Retrieved doc: ", retrieved)

        # 精排        
        reranked = TextReRank.call(
            model="qwen3-rerank",  # 确保该模型在百炼平台可用
            query=query,
            documents=[k.page_content for k in retrieved],
            top_n=3
        )
        rank_retrieved = [retrieved[item["index"]] for item in reranked["output"]["results"]]
        print("【DEBUG】Reranked doc: ", rank_retrieved)
        
        serialized = "\n\n".join(
            f"[{doc.metadata['source']}#{doc.metadata['chunk_id']}] {doc.page_content}"
            for doc in rank_retrieved
        )
        return serialized, rank_retrieved

    return create_agent(
        llm,
        tools=[retrieve_context],
        system_prompt=(
            "你可以使用检索工具获得参考资料。回答时结合检索到的内容，"
            "如有必要可以在答案中简单引用来源标识。"
        ),
    )

## 7. 运行演示

执行以下代码块开始提问。

In [36]:
# 1. 准备并构建向量库
vector_store = build_vector_store()
print('嵌入完成' + '\n')

# 2. 创建 Agent
agent = create_react_agent_wrapper(vector_store)

# 3. 提问
query = "公司的考勤方式是什么？"
print(f"Query: {query}\n" + "-" * 20)

input_payload = {"messages": [{"role": "user", "content": query}]}

for event in agent.stream(input_payload, stream_mode="values"):
    event["messages"][-1].pretty_print()

成功加载 8 个文档到向量库
嵌入完成

Query: 公司的考勤方式是什么？
--------------------
================================ Human Message =================================

公司的考勤方式是什么？
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_3a8eeca8189e42698fcb6470)
 Call ID: call_3a8eeca8189e42698fcb6470
  Args:
    query: 公司考勤方式
【DEBUG】Retrieved doc:  [Document(id='f3f10d35-b5fb-4267-9959-a4fe4b8cb94d', metadata={'source': 'question.txt', 'chunk_id': 0}, page_content='问题:公司的考勤方式是什么？\n答案:公司实行弹性打卡。\n权限: IT组、运维组\n关键词:考勤方式'), Document(id='5b169442-ad30-4829-b157-31b5b74e264f', metadata={'chunk_id': 1, 'source': 'question.txt'}, page_content='问题:公司的考勤方式是什么？\n答案:公司实行固定世界打卡。\n权限: 运营组\n关键词:考勤方式'), Document(id='70dea759-6dfd-43f3-8f1e-1974229ba8a4', metadata={'source': 'question.txt', 'chunk_id': 3}, page_content='问题:忘记打卡怎么办？\n答案:员工缺卡可在缺卡发生后的3个工作日内申请补卡，每人每月限3次，超限不再受理。未补卡者，每缺1次按缺勤半天计算，扣发半天工资。\n权限:\n关键词:因公外出、打卡处理'), Document(id='60e108f8-53ae-4e78-84d6-524e5414f4